In [0]:
CREATE TABLE IF NOT EXISTS sac.customer.customer (
		customer_id STRING NOT NULL,
		last_name STRING,
		first_name STRING,
		signup_date DATE,
		plan_tier STRING,
		plz STRING,
		street STRING,
		city STRING,
		zone STRING,
		ai_used BOOLEAN,
		contract_type STRING,
		autopay_enabled BOOLEAN,
		payment_method STRING,
		monthly_bill DOUBLE,
		speed_tier_mbps INT,
		data_usage_gb_last_month DOUBLE,
		CONSTRAINT customer_pk PRIMARY KEY (customer_id)
	);

-- Edit constraints of table
ALTER TABLE
	sac.customer.customer
DROP CONSTRAINT IF EXISTS
	customer_sd;

ALTER TABLE
	sac.customer.customer
DROP CONSTRAINT IF EXISTS
	customer_mb;

ALTER TABLE
	sac.customer.customer
DROP CONSTRAINT IF EXISTS
	customer_du;

ALTER TABLE
	sac.customer.customer
ADD
	CONSTRAINT customer_sd CHECK (signup_date <= current_date());

ALTER TABLE
	sac.customer.customer
ADD
	CONSTRAINT customer_mb CHECK (monthly_bill >= 0);

ALTER TABLE
	sac.customer.customer
ADD
	CONSTRAINT customer_du
		CHECK (
			data_usage_gb_last_month >= 0
			OR data_usage_gb_last_month IS NULL
		);

MERGE INTO
	sac.customer.customer c
USING (
	SELECT
		*
	FROM
		sac.customer.customer_delete
) d
ON
	c.customer_id = d.customer_id
WHEN MATCHED THEN DELETE;

-- Add table for deleted values
MERGE INTO
	sac.customer.customer c
USING (
	SELECT
		*
	FROM
		sac.customer.customer_delete
) d
ON
	c.customer_id = d.customer_id
WHEN MATCHED THEN DELETE;

-- Fill table with values
WITH f_loc AS (
	SELECT
		customer_id,
		ingestion_time,
		substring_index(address, ',', 1) AS street,
		substr(address, len(street) + 2) AS rest
	FROM
		sac.customer.customer_bronze
),
location AS (
	SELECT
		customer_id,
		ingestion_time,
		CASE
			WHEN
				len(regexp_extract(rest, '([0-9]+)', 1)) = 4
			THEN
				concat('0', regexp_extract(rest, '([0-9]+)', 1))
			ELSE regexp_extract(rest, '([0-9]+)', 1)
		END AS plz,
		replace(street, 'str.', 'straße') AS street,
		CASE
			WHEN rest LIKE '%berlin%' THEN 'Berlin'
			WHEN rest LIKE '%None%' THEN trim(replace(rest, 'None ', ''))
			ELSE substring_index(trim(regexp_extract(rest, '\\d{4,5}\\s+(.+?)\\s*$', 1)), ',', 1)
		END AS city
	FROM
		f_loc
) MERGE INTO
	sac.customer.customer s
USING (
	SELECT
		c.customer_id,
		right(c.customer_name, len(c.customer_name) - charindex(' ', c.customer_name)) AS last_name,
		left(c.customer_name, charindex(' ', c.customer_name)) AS first_name,
		CAST(c.signup_date AS DATE) AS signup_date,
		c.plan_tier,
		left(l.plz, 5) AS plz,
		replace(l.street, 'None', '') AS street,
		l.city AS city,
		p.bundesland AS zone,
		CAST('false' AS BOOLEAN) AS ai_used,
		c.contract_type,
		CASE
			WHEN lower(c.autopay_enabled) LIKE 'true' THEN TRUE
			ELSE FALSE
		END AS autopay_enabled,
		c.payment_method,
		CAST(c.monthly_bill AS DOUBLE) AS monthly_bill,
		CASE
			WHEN regexp_extract(c.plan_tier, '([0-9]+)G', 1) = '1' THEN int(1000)
			ELSE CAST(regexp_extract(c.plan_tier, '([0-9]+)', 1) AS INT)
		END AS speed_tier_mbps,
		CAST(c.data_usage_gb_last_month AS DOUBLE) AS data_usage_gb_last_month
	FROM
		sac.customer.customer_bronze c
			JOIN location l
				ON c.customer_id = l.customer_id
				AND c.ingestion_time = l.ingestion_time
			LEFT JOIN sac.customer.plz_to_state p
				ON CAST(p.plz AS INT) = TRY_CAST(l.plz AS INT)
			LEFT JOIN sac.customer.customer_delete d
				ON c.customer_id = d.customer_id
	WHERE
		d.customer_id IS NULL
	QUALIFY
		row_number() OVER (PARTITION BY c.customer_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
	b.customer_id = s.customer_id
WHEN MATCHED AND
	sha1(
		concat_ws(
			'|',
			b.last_name,
			b.first_name,
			b.signup_date,
			b.plan_tier,
			b.street,
			b.city,
			b.contract_type,
			b.autopay_enabled,
			b.payment_method,
			b.monthly_bill,
			b.speed_tier_mbps,
			b.data_usage_gb_last_month
		)
	)
		!= sha1(
			concat_ws(
				'|',
				s.last_name,
				s.first_name,
				s.signup_date,
				s.plan_tier,
				s.street,
				s.city,
				s.contract_type,
				s.autopay_enabled,
				s.payment_method,
				s.monthly_bill,
				s.speed_tier_mbps,
				s.data_usage_gb_last_month
			)
		)
	THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;